# ADA 1 ChronoNyx adaptive depth

The experiment asks whether the early ADA design can keep adaptive heads,
low-rank conditioning and early exit while meeting a causal decoder contract.

**Primary implementation:** `Ada_experiment_1/Ada.py` (local snapshot SHA-256
`75a309aab4ddadb5b1ae4592e7a43d5a3c3242aef8ae647ecbe1e8af7c2143a9`). The CPU cells below are self-contained.

The first standalone JAX/Flax script calls the model ChronoNyx. It combines
rotary attention with a whole-sequence top-k head selector, a pooled
hypernetwork that produces a low-rank FFN update, a differentiable
cosine-retrieval bank and token-specific exit thresholds.

This is a small PyTorch reconstruction of those mechanisms. The historical
JAX path used a flattened active-token gather and sequence-wide statistics.
The runnable path preserves batch and sequence axes, uses prefix-conditioned
gates and a causal mask, and keeps context available after a token exits.
It uses one memory residual per block. The chronology is source provenance,
not a claim that Torch reproduces JAX initializers or outputs.

## Run this notebook

Use Python 3.11 or later, JupyterLab and a CPU PyTorch build. In an isolated virtual
environment, install `jupyterlab`, `ipykernel` and `torch>=2.6`, then open this
notebook and run all cells. The reference run is deterministic on CPU and neither
reads repository data nor downloads models. Training starts from random weights.

In [1]:
import json
import math
from dataclasses import asdict, dataclass, field, replace
from typing import Any, Callable, Dict, List, Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

SEED = 2031
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cpu")
print(f"PyTorch {torch.__version__}, device={DEVICE}, seed={SEED}")

PyTorch 2.11.0+cpu, device=cpu, seed=2031


## Architecture

Tokens plus learned positions feed a stack of pre-normalized blocks. A block
applies rotary query/key attention, reads a differentiable memory bank, then
uses a hypernetwork to produce rank-four factors for the first FFN projection.
A thresholded token router freezes representations after exit. This implementation
computes attention for every head and evaluates every block over the full tensor,
so top-k masks and early exits are semantic experiments rather than compute savings.

In [2]:
def prefix_mean(x):
    counts = torch.arange(1, x.shape[1] + 1, device=x.device, dtype=x.dtype)
    return x.cumsum(dim=1) / counts[None, :, None]


def apply_rotary(x):
    _, _, length, dim = x.shape
    frequencies = 10000.0 ** (-torch.arange(0, dim, 2, device=x.device) / dim)
    phase = torch.arange(length, device=x.device)[:, None] * frequencies[None, :]
    phase = torch.cat((phase, phase), dim=-1).to(x.dtype)
    first, second = x.chunk(2, dim=-1)
    return x * phase.cos() + torch.cat((-second, first), dim=-1) * phase.sin()


class AdaptiveHeads(nn.Module):
    def __init__(self, width, heads, keep_ratio=0.6, *, bias=False):
        super().__init__()
        assert width % heads == 0 and (width // heads) % 2 == 0
        self.heads = heads
        self.head_dim = width // heads
        self.keep = max(1, int(heads * keep_ratio))
        self.qkv = nn.Linear(width, 3 * width, bias=bias)
        self.proj = nn.Linear(width, width, bias=bias)
        self.gate = nn.Linear(width, heads)

    def forward(self, x, *, historical=False):
        batch, length, width = x.shape
        q, k, v = self.qkv(x).reshape(
            batch, length, 3, self.heads, self.head_dim
        ).permute(2, 0, 3, 1, 4)
        q, k = apply_rotary(q), apply_rotary(k)
        context = x.mean(dim=1, keepdim=True) if historical else prefix_mean(x)
        scores = self.gate(context).sigmoid()
        values, indices = scores.topk(self.keep, dim=-1)
        weights = torch.zeros_like(scores).scatter(-1, indices, values)
        # Dense attention plus a sparse output weight mask is easy to inspect.
        attended = F.scaled_dot_product_attention(q, k, v, is_causal=not historical)
        attended = attended * weights.transpose(1, 2).unsqueeze(-1)
        return self.proj(attended.transpose(1, 2).reshape(batch, length, width))


class HyperFFN(nn.Module):
    def __init__(self, width, d_ff, rank=4):
        super().__init__()
        self.width, self.d_ff, self.rank = width, d_ff, rank
        self.w1 = nn.Parameter(torch.randn(width, d_ff) / math.sqrt(width))
        self.w2 = nn.Parameter(torch.randn(d_ff, width) / math.sqrt(d_ff))
        self.hyper = nn.Sequential(
            nn.Linear(width, 32), nn.ReLU(), nn.Linear(32, rank * (width + d_ff))
        )

    def forward(self, x, *, historical=False):
        context = x.mean(dim=1, keepdim=True) if historical else prefix_mean(x)
        factors = self.hyper(context)
        left, right = factors.split((self.width * self.rank, self.rank * self.d_ff), dim=-1)
        left = left.unflatten(-1, (self.width, self.rank))
        right = right.unflatten(-1, (self.rank, self.d_ff))
        # x @ (W1 + A @ B), evaluated without forming each dense delta matrix.
        low_rank = torch.einsum("btc,btcr->btr", x, left)
        hidden = x @ self.w1 + torch.einsum("btr,btrf->btf", low_rank, right)
        return F.gelu(hidden) @ self.w2

In [3]:
class MemoryReadBank(nn.Module):
    def __init__(self, width, slots=8, memory_dim=8):
        super().__init__()
        self.keys = nn.Parameter(0.01 * torch.randn(slots, memory_dim))
        self.values = nn.Parameter(0.01 * torch.randn(slots, memory_dim))
        self.query = nn.Linear(width, memory_dim, bias=False)
        self.output = nn.Linear(memory_dim, width, bias=False)

    def forward(self, x):
        query = F.normalize(self.query(x), dim=-1)
        scores = (query @ F.normalize(self.keys, dim=-1).T) / 0.1
        return self.output(scores.softmax(dim=-1) @ self.values)

In [4]:
class AdaptiveBlock(nn.Module):
    def __init__(self, width, *, session_writes):
        super().__init__()
        self.norms = nn.ModuleList(nn.LayerNorm(width) for _ in range(3))
        self.attn = AdaptiveHeads(width, heads=4, bias=session_writes)
        self.memory = MemoryReadBank(width)
        self.ffn = HyperFFN(width, d_ff=48)

    def forward(self, x, *, historical=False, session=None):
        x = x + self.attn(self.norms[0](x), historical=historical)
        h = self.norms[1](x)
        x = x + (self.memory(h) if session is None else self.memory(h, session))
        return x + self.ffn(self.norms[2](x), historical=historical)


class EarlyADA(nn.Module):
    def __init__(self, vocab_size=23, width=24, layers=3, *, session_writes=False):
        super().__init__()
        self.tokens = nn.Embedding(vocab_size, width)
        self.positions = nn.Embedding(32, width)
        self.blocks = nn.ModuleList(AdaptiveBlock(width, session_writes=session_writes)
                                    for _ in range(layers))
        self.routers = nn.ModuleList(nn.Linear(width, 1) for _ in range(layers))
        self.norm = nn.LayerNorm(width)
        self.exit_threshold = 0.75

    def forward(self, ids, *, historical=False, sessions=None):
        x = self.tokens(ids) + self.positions(torch.arange(ids.shape[1], device=ids.device))
        active = torch.ones_like(ids, dtype=torch.bool)
        depth = torch.zeros_like(ids)
        for index, (block, router) in enumerate(zip(self.blocks, self.routers), 1):
            session = None if sessions is None else sessions[index - 1]
            candidate = block(x, historical=historical, session=session)
            x = torch.where(active.unsqueeze(-1), candidate, x)
            depth = torch.where(active, index, depth)
            active = active & (router(x).squeeze(-1).sigmoid() <= self.exit_threshold)
        return F.linear(self.norm(x), self.tokens.weight), depth

## Data and optimization

Each sequence starts from a uniformly sampled ID and adds a fixed step modulo
23. Evaluation is drawn from a separate RNG stream under the same distribution.
Small finite sample spaces can contain duplicate rows across these draws.
Cross-entropy uses one-token shifts, and evaluation measures predictive loss and
token accuracy. The reported metrics are fixture outputs.

In [5]:
def arithmetic_sequences(seed, count, length=12, vocab_size=23):
    rng = torch.Generator().manual_seed(seed)
    start = torch.randint(0, vocab_size, (count, 1), generator=rng)
    step = torch.randint(1, 5, (count, 1), generator=rng)
    return (start + step * torch.arange(length)) % vocab_size


train_rows = arithmetic_sequences(SEED + 1, 16)
eval_rows = arithmetic_sequences(SEED + 2, 4)
assert train_rows.dtype == torch.long and eval_rows.shape == (4, 12)
print("Synthetic train/eval shapes:", tuple(train_rows.shape), tuple(eval_rows.shape))

Synthetic train/eval shapes: (16, 12) (4, 12)


In [6]:
SESSION_WRITES = False

def next_token_metrics(logits, ids):
    aligned = logits[:, :-1]
    targets = ids[:, 1:]
    loss = F.cross_entropy(aligned.flatten(0, 1), targets.flatten())
    accuracy = (aligned.argmax(-1) == targets).float().mean()
    return loss, accuracy


model = EarlyADA(session_writes=SESSION_WRITES).to(DEVICE)
model.eval()
with torch.no_grad():
    baseline = next_token_metrics(model(eval_rows)[0], eval_rows)[0].item()

optimizer = torch.optim.AdamW(model.parameters(), lr=0.004)
train_losses = []
model.train()
for step in range(12):
    batch = train_rows[step % 4 * 4 : step % 4 * 4 + 4]
    loss, _ = next_token_metrics(model(batch)[0], batch)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    assert torch.isfinite(grad_norm) and grad_norm > 0
    optimizer.step()
    train_losses.append(loss.item())

model.eval()
with torch.inference_mode():
    logits, depths = model(eval_rows)
    final, accuracy = next_token_metrics(logits, eval_rows)
    assert logits.shape == (4, 12, 23) and torch.isfinite(logits).all()
    assert depths.min() >= 1 and depths.max() <= len(model.blocks)
    continuation = eval_rows[:1, :4].clone()
    for _ in range(3):
        prediction = model(continuation)[0][:, -1].argmax(-1, keepdim=True)
        continuation = torch.cat((continuation, prediction), dim=1)
    assert continuation.shape == (1, 7)

print(json.dumps({
    "validation_targets": eval_rows[:, 1:].numel(),
    "before_ce": round(baseline, 4),
    "after_ce": round(final.item(), 4),
    "accuracy": round(accuracy.item(), 4),
    "first_train_ce": round(train_losses[0], 4),
    "last_train_ce": round(train_losses[-1], 4),
    "sample_ids": continuation[0].tolist(),
}, indent=2))

{
  "validation_targets": 44,
  "before_ce": 13.9341,
  "after_ce": 6.1189,
  "accuracy": 0.0455,
  "first_train_ce": 14.7197,
  "last_train_ce": 4.8755,
  "sample_ids": [
    19,
    22,
    2,
    5,
    0,
    8,
    7
  ]
}


## Correctness gates

Mutating a suffix or another batch row must leave the observed prefix unchanged.
The historical control reintroduces bidirectional attention and whole-sequence
conditioning to expose the original kinds of leakage. It tests those choices
in isolation and does not claim exact reproduction of the legacy implementation.

In [7]:
pair = eval_rows[:2].clone()
changed_future = pair.clone()
changed_future[0, 7:] = (changed_future[0, 7:] + 6) % 23
changed_neighbor = pair.clone()
changed_neighbor[1] = (changed_neighbor[1] + 7) % 23

with torch.inference_mode():
    normal = model(pair)[0]
    future = model(changed_future)[0]
    neighbor = model(changed_neighbor)[0]
    historical = model(pair, historical=True)[0]
    historical_future = model(changed_future, historical=True)[0]
    torch.testing.assert_close(normal[0, :7], future[0, :7], atol=1e-5, rtol=1e-5)
    torch.testing.assert_close(normal[0], neighbor[0], atol=1e-5, rtol=1e-5)
    historical_delta = (historical[0, :7] - historical_future[0, :7]).abs().max().item()

    # Force the first layer to exit and verify that remaining layers cannot
    # replace those token states. This is a routing invariant, not a speed test.
    router = model.routers[0]
    old_weight, old_bias = router.weight.clone(), router.bias.clone()
    try:
        router.weight.zero_()
        router.bias.fill_(20.0)
        _, forced_depth = model(pair)
        assert torch.equal(forced_depth, torch.ones_like(forced_depth))
    finally:
        router.weight.copy_(old_weight)
        router.bias.copy_(old_bias)

assert historical_delta > 1e-5, "The historical control did not expose the intended leakage"
print(f"PASS: prefix invariance, batch isolation and forced early exit. "
      f"Historical control prefix delta={historical_delta:.6f}.")

PASS: prefix invariance, batch isolation and forced early exit. Historical control prefix delta=5.535811.


## Interpretation

A successful run establishes local forward, optimization, short greedy rollout,
and the stated invariants. It does not establish selection quality, wall-clock
gains, paper-level reproducibility, or language-model accuracy. The binary exit
comparison has no task-loss gradient and the fixture does not train an exit policy.